# 2_models/07 — Note-level survival transformer (overall survival)

Trains a landmark-conditioned causal Transformer over each patient's **sequence of note embeddings**
to predict overall survival (`death_met:death`), then evaluates it on held-out patients and compares it
with the existing Cox text and base models. The Cox models in `2_models/04` pool each patient's notes
into one vector; this model reads the notes individually, with their timing.

**Runs after** `1_data/03` (it reads the knitted note store, the survival cohort and the embedding
prediction dataset) and, for the Cox comparison, after `2_models/04` (the full-cohort risk scores).
Nothing downstream depends on it.

**Tier:** a GPU-capable Jupyter session is preferred. CPU works but a 20-trial search takes hours; on CPU
use `NUM_TRIALS = 5` and a smaller `N_EPOCHS`. For batch use, `slurm/launch_note_transformer.sh` runs the
same two modules as one job.

**Landmarks and note availability.** The model scores each patient at landmarks 0, 90, 180 and 365 days
from the anchor, using only notes written at or before the landmark. Note times are fractional days
computed from the raw note and anchor timestamps, never the integer relative-day columns, so a note a few
hours after a landmark is not counted as available at it. The anchor dates carry no time of day, so a
same-day note is treated as available at the landmark only when its timestamp equals the anchor exactly.

**Cohort and split.** With `COHORT = "text"` the patients are the Cox text-model cohort and the 20% test
set is the Cox test split (seed 1234), checked against `text_ipcw_reference.csv.gz`. Validation is carved
from the remaining patients and is the only data used to choose hyperparameters and checkpoints.

**Cox comparison is descriptive.** The Cox risk scores are out-of-fold scores from models trained on
different partitions than the transformer. The comparison therefore scores only patient pairs inside the
same Cox fold, with identical pairs for every model, and weights folds by comparable-pair count. Raw
scores are never pooled across folds. Reported Cox holdout metrics and the transformer's all-test metrics
are kept as separate evaluation scopes.

**Outputs** (under `scheme_results_dir("death_met", ANCHOR)/note_transformer/`): `inputs/` with the
prepared note bundle and `prepare_manifest.json`; `<TAG>/search/` or `<TAG>/single_run/` with checkpoints
and logs; `<TAG>/test/` with `test_metrics.json`, `test_predictions.csv` and `comparison.csv`.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def report_outputs(outputs: list[tuple[str, str]]) -> None:
    """Print size and mtime for each (label, path) that exists."""
    for label, path in outputs:
        if os.path.exists(path):
            mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"[ok     ] {label:<26} {mb:>8.1f} MB   {mtime}")
        else:
            print(f"[missing] {label:<26} {path}")


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    # Polars/Rayon size their pools to every visible core; on a shared node that can exceed
    # RLIMIT_NPROC (ThreadPoolBuildError, EAGAIN). Cap them unless already set by the caller.
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    for _v in ("POLARS_MAX_THREADS", "RAYON_NUM_THREADS"):
        run_env.setdefault(_v, "4")
    run_env.update(env or {})
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

`STAGES` picks what runs, in order: `prepare`, then **one** of `search` or `train`, then `test`.
`search` runs `NUM_TRIALS` random trials; `train` fits the single configuration in
`pipelines/note_transformer/configs/os_config.json`. Running both in one invocation is rejected, because
the test stage must evaluate one explicit checkpoint. To test an earlier run, drop the training stage and
set `CHECKPOINT`; with `CHECKPOINT = None` the runner uses the sole checkpoint that exists for this `TAG`
and refuses to guess when both do.

`OVERWRITE` replaces an existing inputs bundle and the stage outputs. `RESUME` continues a partly
finished search or fit. They are mutually exclusive. Changing `STATIC`, `VAL_FRAC`, `SPLIT_SEED` or
`MAX_PATIENTS` needs `OVERWRITE = True` or a different directory, and a checkpoint trained on one bundle
is rejected on another. `MAX_PATIENTS` builds a smoke bundle in its own directory and skips the Cox split
check.

In [ ]:
import json

import polars as pl

import schemes
from pipelines.note_transformer.prepare_os_inputs import (
    MANIFEST_NAME, default_inputs_dir, notes_array_path, notes_meta_path, surv_path,
)
from pipelines.note_transformer.run_os_transformer import run_dirs

PREPARE_MODULE = "pipelines.note_transformer.prepare_os_inputs"
RUN_MODULE = "pipelines.note_transformer.run_os_transformer"

ANCHOR = "treatment"          # "treatment" or "sequencing"
COHORT = "text"               # "text" (Cox text cohort + Cox test split) or "all"
STATIC = True                 # GENDER, age at anchor, CANCER_TYPE_* as static covariates
VAL_FRAC = 0.2
SPLIT_SEED = 1234             # train/val carve; the Cox test split is fixed
TAG = "default"
STAGES = ["prepare", "search", "test"]   # prepare, then search OR train, then test
NUM_TRIALS = 20               # 5 is a sensible CPU budget
LANDMARKS = None              # None = os_config.json (0, 90, 180, 365); or e.g. "0,90,180,365"
N_EPOCHS = None               # None = os_config.json (30); smaller for smoke runs
DEVICE = "auto"               # auto, cpu or cuda
NUM_THREADS = None            # None = SLURM_CPUS_PER_TASK or cpu_count - 1
OVERWRITE = False
RESUME = False
MAX_PATIENTS = None           # e.g. 300 for a smoke bundle
CHECKPOINT = None             # explicit best_model.pt for a test-only run

unknown = set(STAGES) - {"prepare", "search", "train", "test"}
if unknown:
    raise ValueError(f"unknown stages: {sorted(unknown)}")
if "search" in STAGES and "train" in STAGES:
    raise ValueError("choose search or train, not both: the test stage needs one explicit checkpoint")
if OVERWRITE and RESUME:
    raise ValueError("OVERWRITE and RESUME are mutually exclusive")
if CHECKPOINT and ("search" in STAGES or "train" in STAGES):
    raise ValueError("CHECKPOINT is only for a test-only run; drop the training stage")

INPUTS_DIR = default_inputs_dir(ANCHOR, COHORT, MAX_PATIENTS)
DIRS = run_dirs(ANCHOR, TAG)
COX_EVENT_DIR = schemes.full_cohort_event_dir("death_met", "death", ANCHOR)
COX_RISK_DIR = schemes.full_cohort_risk_dir("death_met", "death", ANCHOR)

print(f"inputs : {INPUTS_DIR}")
print(f"outputs: {DIRS['root']}")
print(f"stages : {STAGES}")

## Preconditions

Required inputs must be present before preparation. The Cox files are optional: without them the
comparison table reports null Cox rows with the reason, and the transformer's own test metrics are
still produced. PyTorch is needed from the search stage on; CUDA is reported, not required.

In [ ]:
import polars as pl

required = [
    ("note metadata", notes_meta_path()),
    ("note embeddings (.npy.zst)", notes_array_path()),
    ("survival cohort", surv_path(ANCHOR)),
    ("embedding prediction df", os.path.join(config.SURV_PATH, schemes.embedding_file("death_met", ANCHOR))),
    ("cancer type features", os.path.join(config.FEATURE_PATH, "cancer_type_df.csv.gz" if ANCHOR == "treatment"
                                          else f"cancer_type_df__{ANCHOR}.csv.gz")),
]
missing = check_inputs(required)

# The raw timestamp columns are required: integer NOTE_TIME_REL_* columns are never a fallback.
anchor_col = {"treatment": "FIRST_TREATMENT_START_DT", "sequencing": "SEQUENCING_DT"}[ANCHOR]
if os.path.exists(notes_meta_path()):
    have = set(pl.scan_parquet(notes_meta_path()).collect_schema().names())
    for col in ("NOTE_DATETIME", anchor_col):
        ok = col in have
        print(f"[{'ok ' if ok else 'MISSING'}] raw timestamp column {col}")
        if not ok:
            missing.append(col)

try:
    import torch
    print(f"torch {torch.__version__}; cuda available: {torch.cuda.is_available()}")
except ImportError:
    print("[MISSING] torch is not importable in this kernel")
    missing.append("torch")

print("\nOptional Cox comparison files:")
report_outputs([
    ("text_risk_scores.csv", os.path.join(COX_RISK_DIR, "text_risk_scores.csv")),
    ("base_risk_scores.csv", os.path.join(COX_RISK_DIR, "base_risk_scores.csv")),
    ("text_test.csv", os.path.join(COX_EVENT_DIR, "text_test.csv")),
    ("base_test.csv", os.path.join(COX_EVENT_DIR, "base_test.csv")),
    ("text_ipcw_reference.csv.gz", os.path.join(COX_EVENT_DIR, "text_ipcw_reference.csv.gz")),
])
if missing:
    raise SystemExit(f"cannot continue, missing: {missing}")

## 1. Prepare inputs

Writes the five-file note bundle (`notes.npy`, `notes.csv`, `outcomes.csv`, `splits.csv`, optional
`patients.csv`) plus `prepare_manifest.json`. An identical, complete bundle is skipped. For `COHORT =
"text"` the Cox test split is checked against the IPCW reference; the manifest's `split_check` records
the result and should read `matched_cox_ipcw_reference` once the Cox full-cohort run exists.

In [ ]:
common_inputs = ["--anchor", ANCHOR, "--inputs-dir", INPUTS_DIR]

if "prepare" in STAGES:
    args = [*common_inputs, "--cohort", COHORT, "--val-frac", str(VAL_FRAC), "--seed", str(SPLIT_SEED),
            "--static" if STATIC else "--no-static"]
    if MAX_PATIENTS:
        args += ["--max-patients", str(MAX_PATIENTS)]
    if OVERWRITE:
        args.append("--overwrite")
    result = run_module(PREPARE_MODULE, args)
    print(f"prepare finished in {result['wall_s'] / 60:.1f} min (exit {result['returncode']})")
    if result["returncode"] != 0:
        raise SystemExit("prepare failed; fix the error above before running later stages")
else:
    print("prepare skipped (not in STAGES)")

manifest_path = os.path.join(INPUTS_DIR, MANIFEST_NAME)
if os.path.exists(manifest_path):
    with open(manifest_path) as handle:
        manifest = json.load(handle)
    print("split_check:", manifest.get("split_check"))
    for key, value in manifest.get("counts", {}).items():
        print(f"  {key:<34} {value}")
else:
    print(f"[missing] {manifest_path}")

## 2. Search / train

`search` draws `NUM_TRIALS` configurations from `configs/os_search_space.json`, trains each, and keeps the
one with the best validation C-index (weighted by eligible patients across landmarks). `train` fits the
single base configuration instead. The test set is not read at this stage.

In [ ]:
runner_common = ["--anchor", ANCHOR, "--inputs-dir", INPUTS_DIR, "--tag", TAG, "--device", DEVICE]
if NUM_THREADS:
    runner_common += ["--num-threads", str(NUM_THREADS)]
if LANDMARKS:
    runner_common += ["--landmarks", str(LANDMARKS)]
if N_EPOCHS:
    runner_common += ["--n-epochs", str(N_EPOCHS)]
if OVERWRITE:
    runner_common.append("--overwrite")
if RESUME:
    runner_common.append("--resume")

TRAINED_CHECKPOINT = None   # set only by a stage that succeeds in this invocation
for stage, subdir in (("search", "search"), ("train", "single_run")):
    if stage not in STAGES:
        continue
    args = [*runner_common, "--stage", stage]
    if stage == "search":
        args += ["--num-trials", str(NUM_TRIALS)]
    result = run_module(RUN_MODULE, args)
    print(f"{stage} finished in {result['wall_s'] / 60:.1f} min (exit {result['returncode']})")
    if result["returncode"] != 0:
        raise SystemExit(f"{stage} failed; the test stage will not run")
    TRAINED_CHECKPOINT = os.path.join(DIRS[subdir], "best_model.pt")
    print("checkpoint:", TRAINED_CHECKPOINT)
if TRAINED_CHECKPOINT is None:
    print("no training stage requested")

## 3. Held-out test

Evaluates exactly one checkpoint on the test patients: the one trained in this invocation, otherwise
`CHECKPOINT`, otherwise the runner's sole-candidate rule. An older search checkpoint is never preferred
over a model trained just now. Do not retune hyperparameters from these results.

In [ ]:
if "test" in STAGES:
    args = [*runner_common, "--stage", "test"]
    explicit = TRAINED_CHECKPOINT or CHECKPOINT
    if explicit:
        args += ["--checkpoint", explicit]
    result = run_module(RUN_MODULE, args)
    print(f"test finished in {result['wall_s'] / 60:.1f} min (exit {result['returncode']})")
    if result["returncode"] != 0:
        raise SystemExit("test failed")
else:
    print("test skipped (not in STAGES)")

## Summary

Everything below is read from disk, so it also describes earlier runs of the same `TAG`. The comparison
table has three evaluation scopes: `within_cox_folds` (transformer and Cox on the same pairs, the only
like-for-like comparison here), `all_test` (the transformer on every eligible test patient) and
`reported_cox_holdout` (the Cox models' own landmark-0 test metrics). Compare rows within a scope, and
read `n_comparable_pairs` and the exclusion counts before the C-index.

In [ ]:
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(60)

metrics_path = os.path.join(DIRS["test"], "test_metrics.json")
if os.path.exists(metrics_path):
    with open(metrics_path) as handle:
        report = json.load(handle)
    print("model id   :", report["model_id"])
    print("checkpoint :", report["checkpoint_path"])
    weighted_score = report["weighted_test_c_index"]
    weighted_label = "unavailable" if weighted_score is None else f"{weighted_score:.4f}"
    print(f"weighted test C-index: {weighted_label}")
    print(pl.DataFrame(report["landmark_metrics"]).select(
        "landmark", "n_eligible", "n_events", "c_index", "c_index_reason"))
else:
    print(f"[missing] {metrics_path}")

search_csv = os.path.join(DIRS["search"], "hparam_search_results.csv")
if os.path.exists(search_csv):
    board = pl.read_csv(search_csv)
    keep = [c for c in ("trial_id", "status", "validation_score", "best_epoch", "runtime_seconds",
                        "config_json") if c in board.columns]
    print("\nTop search trials by validation C-index:")
    print(board.filter(pl.col("status") == "success").sort("validation_score", descending=True)
          .select(keep).head(10) if "status" in board.columns else board.select(keep).head(10))
else:
    print(f"\n[missing] {search_csv}")

comparison_csv = os.path.join(DIRS["test"], "comparison.csv")
if os.path.exists(comparison_csv):
    comparison = pl.read_csv(comparison_csv)
    for scope in comparison["evaluation_scope"].unique(maintain_order=True):
        print(f"\nEvaluation scope: {scope}")
        print(comparison.filter(pl.col("evaluation_scope") == scope).drop("evaluation_scope"))
else:
    print(f"\n[missing] {comparison_csv}")

report_outputs([
    ("prepare manifest", os.path.join(INPUTS_DIR, MANIFEST_NAME)),
    ("search summary", os.path.join(DIRS["search"], "search_summary.json")),
    ("search checkpoint", os.path.join(DIRS["search"], "best_model.pt")),
    ("single-run checkpoint", os.path.join(DIRS["single_run"], "best_model.pt")),
    ("test metrics", os.path.join(DIRS["test"], "test_metrics.json")),
    ("test predictions", os.path.join(DIRS["test"], "test_predictions.csv")),
    ("comparison", os.path.join(DIRS["test"], "comparison.csv")),
])